中国仓鼠细胞 (CHO) 未知外源工程基因挖掘指南 (优化版)

核心算法思想：宿主序列减除法

在不知道外源插入片段是什么的情况下，我们不能用常规的比对法，而必须采用“排除法”。

关键步骤 (Workflow)

第一步：宿主背景减除 (Host Subtraction)

动作：将您的测序数据 (FASTQ) 比对到最新的中国仓鼠参考基因组（如 NCBI 的 Cricetulus_griseus_picr 或 CriGri-PICR）。

工具：BWA-MEM 或 Bowtie2。

目的：把属于仓鼠本身的 reads 全部标记出来。

第二步：提取“孤儿”与“嵌合” Reads

动作：使用 samtools 专门提取出两类 Reads：

Unmapped reads：双端 read 都没有比对到仓鼠基因组上的（纯外源序列）。

Discordant/Split reads：一端比对到了仓鼠上，另一端没比对上（这是外源基因插入位点的边缘序列）。

命令示例：samtools view -b -f 4 (提取未比对 reads)。

第三步：外源序列从头组装 (De novo Assembly)

动作：把第二步提取出来的这些“非仓鼠” reads，直接进行从头拼接，拼接成较长的序列（Contigs）。

工具：强烈推荐使用 MEGAHIT（极度省内存），或者 SPAdes。

⚠️ 致命警告：千万不要对全基因组的原始 FASTQ 直接进行从头组装！ 仓鼠基因组有 $\approx 2.4\text{ Gb}$，全基因组拼接至少需要 $500\text{ GB} - 1\text{ TB}$ 内存。您的 $128\text{ GB}$ 内存只能用来拼接第二步提取出来的“几百万条未比对 reads”。

第四步：盲测身份鉴定 (轻量化在线/远程 BLAST)

由于第三步拼接出来的非仓鼠 Contigs 数量极少（通常只有个位数），下载 $150\text{ GB}$ 的本地 nt 库是完全没有必要的。我们有以下两个更高效、完全零磁盘占用的替代方案：

方案 A（极力推荐：NCBI Web BLAST）：

动作：直接用 cat 查看组装出来的 final.contigs.fa。由于只有几条序列，直接复制它们，粘贴到 NCBI BLASTn 网页端。

优势：完全不占本地磁盘空间。网页版能提供极度直观的图形化对齐图，您可以一眼看出这段外源序列是否包含特定的启动子（如 CMV）、抗性标记（如 Puro/Neo）或载体骨架（如 pCDNA3.1）。

方案 B（命令行远程比对：blastn -remote）：

动作：如果您必须在 Positron 终端完成自动化流程，可以直接调用本地 blastn 程序的 -remote 参数，让本地客户端把序列打包发送给 NCBI 服务器进行云端检索。

命令示例：

blastn -query final.contigs.fa -db nt -remote -outfmt 6 -out blast_results.txt


优势：同样零磁盘占用，直接在终端生成标准的表格化比对结果，且无需维护本地数据库。

硬件资源与时间消耗评估 (基于单样本 $30\text{X}$ WGS 测序)

假设您的单个仓鼠样本数据量约为 $90\text{ GB}$ 的 FASTQ 文件。结合您的硬件（$64$ 线程，$128\text{ GB}$ 内存，读取速度 $750\text{ MB/s}$，写入速度 $250\text{ MB/s}$）：

分析阶段

工具

计算特点与瓶颈分析

预估耗时

内存峰值

本地存储消耗

1. 宿主比对

bwa-mem

I/O 严重受限。64线程计算极快，但生成巨大 BAM 文件时，会被 $250\text{ MB/s}$ 写入速度拖累。

5 - 7 小时

$\approx 15\text{ GB}$

产生 $\approx 100\text{ GB}$ BAM 缓存

2. 提取未比对 reads

samtools

I/O 受限。需要从巨大的 BAM 中过滤并写出 FASTQ。

1.5 - 2 小时

$< 2\text{ GB}$

$\approx 2 - 5\text{ GB}$ (数据量极小)

3. 从头组装

MEGAHIT

纯计算受限。因为只拼接少量剩余 reads，数据量很小。

0.5 - 1 小时

$10 - 20\text{ GB}$

$< 1\text{ GB}$

4. 在线/远程 BLAST

NCBI Web / blastn -remote

完全受限于网络延迟。本地无任何 CPU 和 I/O 开销，数据直接上云。

几分钟

$0\text{ GB}$

$0\text{ GB}$ (原本地方案需 $150\text{ GB}$)

总计



单个 WGS 样本的端到端盲测发现

7 - 10 小时

绝对安全 (< 20GB)

约 110 GB (极大缓解 SSD 写入压力)

落地实操与 AI 协同建议

这个分析流程看似简单，但由于涉及非常多的中间文件过滤和管道操作（Pipes），手写 bash 很容易产生数以百 GB 计的临时垃圾文件，撑爆您的外外接 SSD。

建议您这样指挥您的 AI Agent (Aider / Claude Code CLI)：

编写规划书：新建一个 find_insertions.md，把上面的四步逻辑贴进去。

强制要求使用管道 (Piping)：让 AI 在写 bash 脚本时，把步骤 1 和步骤 2 连起来写。例如使用 bwa mem ... | samtools view -b -f 4 ... > unmapped.bam。绝对不要让它先生成一个全基因组的 $100\text{ GB}$ BAM 文件，然后再去过滤。这不仅省去了几个小时的写入时间，还省了 $100\text{ GB}$ 硬盘空间。

让 AI 写 MEGAHIT 脚本：提醒 AI，您的服务器只有 $128\text{ GB}$ 内存，让它在命令中加入 -m 0.5（限制 MEGAHIT 最多使用系统 $50\%$ 的内存）。